### Update advance

In [1]:
# 1. Instalasi (jika perlu)
# pip install pandas scikit-learn langdetect textblob transformers torch

import pandas as pd
from langdetect import detect
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.decomposition import LatentDirichletAllocation
from textblob import TextBlob
from transformers import pipeline

# 2. Load & deteksi bahasa (tangani NaN)
df = pd.read_csv('data_stemm.csv')
def detect_lang_safe(text):
    if not isinstance(text, str) or not text.strip(): return 'unknown'
    try: return detect(text)
    except: return 'unknown'

df['lang'] = df['stemmed'].apply(detect_lang_safe)
df = df[df['lang'].isin(['en','id'])].copy()
df = df[df['stemmed'].str.strip() != '']

# 3. Siapkan sentiment analyzers
# English via TextBlob
def sentiment_en(txt):
    p = TextBlob(txt).sentiment.polarity
    return 'positive' if p > 0.1 else 'negative' if p < -0.1 else 'neutral'

# Indonesian via Roberta classifier
sent_id = pipeline(
    "sentiment-analysis",
    model="w11wo/indonesian-roberta-base-sentiment-classifier",
    tokenizer="w11wo/indonesian-roberta-base-sentiment-classifier"
)
def sentiment_id(txt):
    out = sent_id(txt[:512])[0]
    lbl = out['label'].lower()
    if 'neg' in lbl: return 'negative'
    if 'pos' in lbl: return 'positive'
    return 'neutral'

# 4. Siapkan emotion analyzers untuk EN & ID
# (a) English Emotion
emotion_en = pipeline(
    "text-classification",
    model="j-hartmann/emotion-english-distilroberta-base",
    return_all_scores=True
)
# (b) Indonesian Emotion (Plutchik)
emotion_id = pipeline(
    "text-classification",
    model="Aardiiiiy/NusaBERT-base-Indonesian-Plutchik-emotion-analysis-v2",
    tokenizer="Aardiiiiy/NusaBERT-base-Indonesian-Plutchik-emotion-analysis-v2",
    return_all_scores=True
)

# 5. Mapping label ke 5 emosi target
en_map = {
    'joy': 'happy', 'anger': 'angry', 'fear': 'scared',
    'sadness': 'sad', 'surprise': 'surprised'
}
id_map = {
    'joy': 'happy', 'anger': 'angry', 'fear': 'scared',
    'sadness': 'sad', 'surprise': 'surprised'
}

# 6. Fungsi unified emotion
def emotion_multi(text, lang):
    txt = text[:512]
    try:
        if lang == 'en':
            scores = emotion_en(txt)[0]
            top = max(scores, key=lambda x: x['score'])
            return en_map.get(top['label'].lower(), 'neutral')
        else:
            scores = emotion_id(txt)[0]
            top = max(scores, key=lambda x: x['score'])
            return id_map.get(top['label'].lower(), 'neutral')
    except:
        return 'neutral'

# 7. Fungsi utama analisis per bahasa (LDA + Sentiment + Emotion)
def analyze_language(df, lang_code, n_topics=5, n_top_words=10):
    sub = df[df['lang'] == lang_code].copy()
    texts = sub['stemmed'].tolist()

    # a) LDA Topic Modeling
    vect = CountVectorizer(max_df=0.8, min_df=5)
    dtm = vect.fit_transform(texts)
    lda = LatentDirichletAllocation(n_components=n_topics, random_state=42)
    lda.fit(dtm)

    # b) Dominant Topic
    topic_dist = lda.transform(dtm)
    sub['dominant_topic'] = topic_dist.argmax(axis=1)

    # c) Sentiment
    if lang_code == 'en':
        sub['sentiment'] = sub['stemmed'].apply(sentiment_en)
    else:
        sub['sentiment'] = sub['stemmed'].apply(sentiment_id)

    # d) Emotion (multi-bahasa)
    sub['emotion'] = sub.apply(lambda row: emotion_multi(row['stemmed'], row['lang']), axis=1)

    # e) Summary Sentiment per Topic
    sent_summary = (sub.groupby(['dominant_topic', 'sentiment'])
                     .size().unstack(fill_value=0))
    sent_summary['topic_sentiment'] = sent_summary.idxmax(axis=1)

    # f) Summary Emotion per Topic
    emo_summary = (sub.groupby(['dominant_topic', 'emotion'])
                   .size().unstack(fill_value=0))

    # g) Cetak top words + sentimen dominan
    feat = vect.get_feature_names_out()
    print(f"\n=== Language: {lang_code} ===")
    for t in range(n_topics):
        top_idx = lda.components_[t].argsort()[:-n_top_words-1:-1]
        topw = [feat[i] for i in top_idx]
        sent_lbl = sent_summary.loc[t, 'topic_sentiment'] if t in sent_summary.index else 'N/A'
        print(f"Topic {t+1} ({sent_lbl}): {', '.join(topw)}")

    # h) Cetak ringkasan
    print("\nDistribusi Sentimen per Topik:")
    print(sent_summary)
    print("\nDistribusi Emosi per Topik:")
    print(emo_summary)

    return sub, sent_summary, emo_summary

# 8. Jalankan analisis untuk EN & ID
sub_en, sent_en, emo_en = analyze_language(df, 'en', n_topics=3, n_top_words=10)
sub_id, sent_id, emo_id = analyze_language(df, 'id', n_topics=3, n_top_words=10)


c:\Users\HAJRAN\AppData\Local\Programs\Python\Python311\Lib\site-packages\torch\_utils.py:776: UserWarning: TypedStorage is deprecated. It will be removed in the future and UntypedStorage will be the only storage class. This should only matter to you if you are using storages directly.  To access UntypedStorage directly, use tensor.untyped_storage() instead of tensor.storage()
  return self.fget.__get__(instance, owner)()
c:\Users\HAJRAN\AppData\Local\Programs\Python\Python311\Lib\site-packages\transformers\pipelines\text_classification.py:104: UserWarning: `return_all_scores` is now deprecated,  if want a similar functionality use `top_k=None` instead of `return_all_scores=True` or `top_k=1` instead of `return_all_scores=False`.
  warnings.warn(
Special tokens have been added in the vocabulary, make sure the associated word embeddings are fine-tuned or trained.
Special tokens have been added in the vocabulary, make sure the associated word embeddings are fine-tuned or trained.



=== Language: en ===
Topic 1 (positive): app, ad, lesson, use, heart, learn, get, practic, im, time
Topic 2 (positive): learn, languag, app, word, like, duolingo, use, make, good, new
Topic 3 (positive): get, use, duolingo, app, pay, learn, free, subscript, lesson, ad

Distribusi Sentimen per Topik:
sentiment       negative  neutral  positive topic_sentiment
dominant_topic                                             
0                     41      124       197        positive
1                     41      130       288        positive
2                     13       60       106        positive

Distribusi Emosi per Topik:
emotion         angry  happy  neutral  sad  scared  surprised
dominant_topic                                               
0                  46     93       16  159       5         43
1                  30    210       59  111       7         42
2                  14     45       17   78       4         21

=== Language: id ===
Topic 1 (positive): nya, ajar, iklan,

### DASHBOARD

In [ ]:
import dash
from dash import html, dcc
from dash.dependencies import Input, Output
from wordcloud import WordCloud
from io import BytesIO
import base64
import plotly.express as px

# 9. Siapkan data untuk Dash Dashboard (English example)
# Document counts per topic
doc_counts = sub_en['dominant_topic'].value_counts().sort_index()
doc_df = pd.DataFrame({
    'Topic': [f"Topic {i+1}" for i in doc_counts.index],
    'Count': doc_counts.values
})

# Sentiment distribution per topic
sent_df = sent_en.copy().reset_index()
sent_df['Topic'] = sent_df['dominant_topic'].map(lambda i: f"Topic {i+1}")
sent_long = sent_df.melt(
    id_vars='Topic',
    value_vars=['positive','neutral','negative'],
    var_name='Sentiment',
    value_name='Count'
)

# Emotion distribution per topic
emo_df = emo_en.copy().reset_index()
emo_df['Topic'] = emo_df['dominant_topic'].map(lambda i: f"Topic {i+1}")
emo_long = emo_df.melt(
    id_vars='Topic',
    value_vars=[c for c in emo_df.columns if c not in ['dominant_topic']],
    var_name='Emotion',
    value_name='Count'
)

# Pre-generate WordClouds
def create_wc_img(lda_model, vect, topic_idx, n_top_words=20):
    freqs = {
        vect.get_feature_names_out()[i]: lda_model.components_[topic_idx][i]
        for i in lda_model.components_[topic_idx].argsort()[:-n_top_words-1:-1]
    }
    wc = WordCloud(width=600, height=300, background_color='white')
    wc = wc.generate_from_frequencies(freqs)
    buf = BytesIO()
    wc.to_image().save(buf, format='PNG')
    return "data:image/png;base64," + base64.b64encode(buf.getvalue()).decode()

# Define lda_en and vect_en using the existing analysis results
lda_en = LatentDirichletAllocation(n_components=5, random_state=42)
vect_en = CountVectorizer(max_df=0.8, min_df=5)
dtm_en = vect_en.fit_transform(sub_en['stemmed'].tolist())
lda_en.fit(dtm_en)

wordclouds_en = {
    f"Topic {i+1}": create_wc_img(lda_en, vect_en, i)
    for i in range(5)
}

# 10. Build Dash App
app = dash.Dash(__name__)
app.layout = html.Div([
    html.H1("Topic, Sentiment & Emotion Dashboard (EN)"),

    dcc.Graph(
        id='doc-counts',
        figure=px.bar(doc_df, x='Topic', y='Count', title="Jumlah Dokumen per Topik")
    ),

    dcc.Graph(
        id='sentiment-dist',
        figure=px.bar(sent_long, x='Topic', y='Count', color='Sentiment', title="Distribusi Sentimen per Topik", barmode='stack')
    ),

    dcc.Graph(
        id='emotion-dist',
        figure=px.bar(emo_long, x='Topic', y='Count', color='Emotion', title="Distribusi Emosi per Topik", barmode='stack')
    ),

    html.Div([
        html.Label("Pilih Topik untuk WordCloud:"),
        dcc.Dropdown(
            id='topic-dropdown',
            options=[{'label': f"Topic {i+1}", 'value': f"Topic {i+1}"} for i in range(5)],
            value='Topic 1'
        ),
        html.Img(id='wc-image', style={'width':'60%', 'marginTop':'20px'})
    ], style={'width':'50%', 'margin':'auto'})
])

@app.callback(
    Output('wc-image', 'src'),
    Input('topic-dropdown', 'value')
)
def update_wc(selected_topic):
    return wordclouds_en[selected_topic]

if __name__ == '__main__':
    app.run(debug=True)

c:\Users\HAJRAN\AppData\Local\Programs\Python\Python311\Lib\site-packages\plotly\express\_core.py:2065: FutureWarning:

When grouping with a length-1 list-like, you will need to pass a length-1 tuple to get_group in a future version of pandas. Pass `(name,)` instead of `name` to silence this warning.

c:\Users\HAJRAN\AppData\Local\Programs\Python\Python311\Lib\site-packages\plotly\express\_core.py:2065: FutureWarning:

When grouping with a length-1 list-like, you will need to pass a length-1 tuple to get_group in a future version of pandas. Pass `(name,)` instead of `name` to silence this warning.

